In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
methods_names = {
    "dist_linguistic_confidence": "Dist. Ling. Conf.",
    "dist_semantic_uncertainty": "Dist. Semantic Unc.",
    "dist_lnll": "Dist. Token Prob"
}

dataset_size = {
    "mmlu": 116000,
    "squadv2": 130000,
    "truthful_qa": 817
}

dataset_map = {
    "mmlu": "MMLU",
    "squadv2": "SQuAD2.0",
    "truthful_qa": "TruthfulQA"
}

model_name_map = {
    "Llama-3.1-8B-Instruct": "Llama-3.1-8B-Inst.",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B-Inst.",
    "Qwen2.5-7B-Instruct": "Qwen2.5-7B-Inst.",
    "Qwen3-8B": "Qwen3-8B-Inst.",
    "Mistral-7B-Instruct-v0.3": "Mistral-7B-Inst.",
    "gpt-oss-20b": "GPT-OSS-20B",
}

# Cross domain data

In [3]:
prompt_type = "direct_qa"

In [4]:
results_dir = f"/hdd/ivny/{prompt_type}_cross_domain_calibration"

# list all leaf nodes in results_dir
leaf_dirs = []
for root, dirs, files in os.walk(results_dir):
    if not dirs:  # if there are no subdirectories, it's a leaf node
        leaf_dirs.append(root)

all_records = []
for leaf_dir in leaf_dirs:
    _, _, _, _, dataset_name, _, model_name = leaf_dir.split("/")
    training_set, test_set = dataset_name.split("--")
    if os.path.exists(os.path.join(leaf_dir, "calibration_performance.csv")):
        csv_data = pd.read_csv(os.path.join(leaf_dir, "calibration_performance.csv"), index_col=0)
        csv_records = csv_data.transpose().to_dict("records")

        for record in csv_records:
            record["model"] = model_name_map.get(model_name, model_name)
            record["training_set"] = dataset_map.get(training_set, training_set)
            record["test_set"] = dataset_map.get(test_set, test_set)
            record["dataset_size"] = dataset_size.get(test_set, None)

        all_records.extend(csv_records)
    else:
        print(model_name, dataset_name, "missing calibration_performance.csv")

In [5]:
df = pd.DataFrame(all_records).drop(columns=["model", "dataset_size"])
dataset_weighted_average_mean = df.groupby(["training_set", "test_set"]).mean().reset_index()

In [6]:
dataset_weighted_average_mean

,training_set,test_set,original_lc_generalised_ECE,original_lc_faithfulness_divergence,original_lc_ece_mean,original_lc_dAUROC,original_lc_auroc_mean,original_tp_generalised_ECE,original_tp_faithfulness_divergence,original_tp_ece_mean,...,calibrated_tp_rewritten_lc_generalised_ECE,calibrated_tp_rewritten_lc_faithfulness_divergence,calibrated_tp_rewritten_lc_ece_mean,calibrated_tp_rewritten_lc_dAUROC,calibrated_tp_rewritten_lc_auroc_mean,calibrated_su_rewritten_lc_generalised_ECE,calibrated_su_rewritten_lc_faithfulness_divergence,calibrated_su_rewritten_lc_ece_mean,calibrated_su_rewritten_lc_dAUROC,calibrated_su_rewritten_lc_auroc_mean
0,MMLU,SQuAD2.0,0.361068,2.666756,0.348688,0.520925,0.528579,0.252716,12.503721,0.247170,...,0.217936,0.998843,0.192646,0.556200,0.576259,0.220295,1.033671,0.205532,0.585413,0.602906
1,MMLU,TruthfulQA,0.388494,2.348493,0.382936,0.569050,0.604822,0.181605,6.851037,0.176229,...,0.228146,0.721413,0.198303,0.574025,0.613780,0.250687,0.936188,0.240624,0.602200,0.631587
2,SQuAD2.0,MMLU,0.244017,1.777057,0.193088,0.516613,0.524001,0.222447,92.939325,0.198355,...,0.135304,0.549224,0.091710,0.538900,0.566447,0.114570,0.446116,0.088637,0.611650,0.662186
3,SQuAD2.0,TruthfulQA,0.388512,2.348493,0.382936,0.563750,0.604822,0.181510,6.851037,0.176229,...,0.141877,0.498328,0.082309,0.558875,0.607168,0.190069,0.656468,0.174490,0.600275,0.634786
4,TruthfulQA,MMLU,0.243987,1.777057,0.193088,0.517700,0.524001,0.222471,92.939325,0.198355,...,0.145053,0.490595,0.102225,0.530800,0.549978,0.159034,0.461197,0.127812,0.570200,0.616766
5,TruthfulQA,SQuAD2.0,0.361034,2.666756,0.348688,0.523000,0.528579,0.252781,12.503721,0.247170,...,0.175389,0.725119,0.136122,0.543775,0.555117,0.156066,0.602174,0.107956,0.531375,0.551859


In [7]:
dataset_weighted_average_mean.columns.tolist()

['training_set',
 'test_set',
 'original_lc_generalised_ECE',
 'original_lc_faithfulness_divergence',
 'original_lc_ece_mean',
 'original_lc_dAUROC',
 'original_lc_auroc_mean',
 'original_tp_generalised_ECE',
 'original_tp_faithfulness_divergence',
 'original_tp_ece_mean',
 'original_tp_dAUROC',
 'original_tp_auroc_mean',
 'original_su_generalised_ECE',
 'original_su_faithfulness_divergence',
 'original_su_ece_mean',
 'original_su_dAUROC',
 'original_su_auroc_mean',
 'calibrated_lc_generalised_ECE',
 'calibrated_lc_faithfulness_divergence',
 'calibrated_lc_ece_mean',
 'calibrated_lc_dAUROC',
 'calibrated_lc_auroc_mean',
 'calibrated_tp_generalised_ECE',
 'calibrated_tp_faithfulness_divergence',
 'calibrated_tp_ece_mean',
 'calibrated_tp_dAUROC',
 'calibrated_tp_auroc_mean',
 'calibrated_su_generalised_ECE',
 'calibrated_su_faithfulness_divergence',
 'calibrated_su_ece_mean',
 'calibrated_su_dAUROC',
 'calibrated_su_auroc_mean',
 'calibrated_lc_rewritten_lc_generalised_ECE',
 'calibrate

In [8]:
pct = True

In [9]:
fd_improvement_df = dataset_weighted_average_mean[["training_set", "test_set"]].copy()


fd_improvement_df["Linguistic Confidence"] = (dataset_weighted_average_mean["calibrated_lc_rewritten_lc_faithfulness_divergence"] - dataset_weighted_average_mean["original_lc_faithfulness_divergence"]) 
fd_improvement_df["Token Probability"] = (dataset_weighted_average_mean["calibrated_tp_rewritten_lc_faithfulness_divergence"] - dataset_weighted_average_mean["original_lc_faithfulness_divergence"]) 
fd_improvement_df["Semantic Uncertainty"] = (dataset_weighted_average_mean["calibrated_su_rewritten_lc_faithfulness_divergence"] - dataset_weighted_average_mean["original_lc_faithfulness_divergence"]) 

if pct:
    fd_improvement_df["Linguistic Confidence"] = fd_improvement_df["Linguistic Confidence"] / dataset_weighted_average_mean["original_lc_faithfulness_divergence"]
    fd_improvement_df["Token Probability"] = fd_improvement_df["Token Probability"] / dataset_weighted_average_mean["original_lc_faithfulness_divergence"]
    fd_improvement_df["Semantic Uncertainty"] = fd_improvement_df["Semantic Uncertainty"] / dataset_weighted_average_mean["original_lc_faithfulness_divergence"]

fd_improvement_df.to_dict("records")

[{'training_set': 'MMLU',
  'test_set': 'SQuAD2.0',
  'Linguistic Confidence': -0.6345056446779769,
  'Token Probability': -0.6254463481305345,
  'Semantic Uncertainty': -0.6123864476390748},
 {'training_set': 'MMLU',
  'test_set': 'TruthfulQA',
  'Linguistic Confidence': -0.5913115003100472,
  'Token Probability': -0.6928185820075983,
  'Semantic Uncertainty': -0.6013665686338633},
 {'training_set': 'SQuAD2.0',
  'test_set': 'MMLU',
  'Linguistic Confidence': -0.7064812213300623,
  'Token Probability': -0.6909362653697121,
  'Semantic Uncertainty': -0.7489577945616173},
 {'training_set': 'SQuAD2.0',
  'test_set': 'TruthfulQA',
  'Linguistic Confidence': -0.7064132213628966,
  'Token Probability': -0.7878094321183975,
  'Semantic Uncertainty': -0.7204726799613677},
 {'training_set': 'TruthfulQA',
  'test_set': 'MMLU',
  'Linguistic Confidence': -0.7100115418249231,
  'Token Probability': -0.7239286908248913,
  'Semantic Uncertainty': -0.7404715605547958},
 {'training_set': 'TruthfulQA'

In [10]:
ece_improvement_df  = dataset_weighted_average_mean[["training_set", "test_set"]].copy()

ece_improvement_df["Linguistic Confidence"] = (dataset_weighted_average_mean["calibrated_lc_rewritten_lc_generalised_ECE"] - dataset_weighted_average_mean["original_lc_generalised_ECE"]) 
ece_improvement_df["Token Probability"] = (dataset_weighted_average_mean["calibrated_tp_rewritten_lc_generalised_ECE"] - dataset_weighted_average_mean["original_lc_generalised_ECE"]) 
ece_improvement_df["Semantic Uncertainty"] = (dataset_weighted_average_mean["calibrated_su_rewritten_lc_generalised_ECE"] - dataset_weighted_average_mean["original_lc_generalised_ECE"]) 

if pct:
    ece_improvement_df["Linguistic Confidence"] = ece_improvement_df["Linguistic Confidence"] / dataset_weighted_average_mean["original_lc_generalised_ECE"]
    ece_improvement_df["Token Probability"] = ece_improvement_df["Token Probability"] / dataset_weighted_average_mean["original_lc_generalised_ECE"]
    ece_improvement_df["Semantic Uncertainty"] = ece_improvement_df["Semantic Uncertainty"] / dataset_weighted_average_mean["original_lc_generalised_ECE"]

ece_improvement_df.to_dict("records")

[{'training_set': 'MMLU',
  'test_set': 'SQuAD2.0',
  'Linguistic Confidence': -0.35946419842547567,
  'Token Probability': -0.39641144708039977,
  'Semantic Uncertainty': -0.3898798782662345},
 {'training_set': 'MMLU',
  'test_set': 'TruthfulQA',
  'Linguistic Confidence': -0.2923963494185573,
  'Token Probability': -0.41274116709020525,
  'Semantic Uncertainty': -0.3547215722367931},
 {'training_set': 'SQuAD2.0',
  'test_set': 'MMLU',
  'Linguistic Confidence': -0.27897605869593733,
  'Token Probability': -0.4455146408143513,
  'Semantic Uncertainty': -0.5304835647053388},
 {'training_set': 'SQuAD2.0',
  'test_set': 'TruthfulQA',
  'Linguistic Confidence': -0.47631347265291774,
  'Token Probability': -0.6348201169483187,
  'Semantic Uncertainty': -0.5107759416079183},
 {'training_set': 'TruthfulQA',
  'test_set': 'MMLU',
  'Linguistic Confidence': -0.1284544924092972,
  'Token Probability': -0.40549015156100227,
  'Semantic Uncertainty': -0.3481884343070705},
 {'training_set': 'Truth

# In domain diagonal data

In [11]:
results_dir = f"/hdd/ivny/{prompt_type}_in_domain_calibration"

# list all leaf nodes in results_dir
leaf_dirs = []
for root, dirs, files in os.walk(results_dir):
    if not dirs:  # if there are no subdirectories, it's a leaf node
        leaf_dirs.append(root)

all_records = []
for leaf_dir in leaf_dirs:
    _, _, _, _, dataset_name, _, model_name = leaf_dir.split("/")
    if os.path.exists(os.path.join(leaf_dir, "calibration_performance.csv")):
        csv_data = pd.read_csv(os.path.join(leaf_dir, "calibration_performance.csv"), index_col=0)
        csv_records = csv_data.transpose().to_dict("records")

        for record in csv_records:
            record["dataset"] = dataset_map.get(dataset_name, dataset_name)
            record["model"] = model_name_map.get(model_name, model_name)
            record["dataset_size"] = dataset_size.get(dataset_name, None)

        all_records.extend(csv_records)
    else:
        print(model_name, dataset_name, "missing calibration_performance.csv")

In [12]:
full_df = pd.DataFrame(all_records)
full_df = full_df.drop(columns=["model", "dataset_size"]).groupby("dataset").mean().reset_index()

in_domain_ece = []
in_domain_fd = []

for _, row in full_df.iterrows():
    if pct:
        ece = {
            'training_set': row['dataset'],
            'test_set': row['dataset'],
            'Linguistic Confidence': (row["calibrated_lc_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE']) / row['original_lc_generalised_ECE'],
            'Token Probability': (row["calibrated_tp_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE']) / row['original_lc_generalised_ECE'],
            'Semantic Uncertainty': (row["calibrated_su_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE']) / row['original_lc_generalised_ECE']
        }

        fd = {
            'training_set': row['dataset'],
            'test_set': row['dataset'],
            'Linguistic Confidence': (row["calibrated_lc_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence']) / row['original_lc_faithfulness_divergence'],
            'Token Probability': (row["calibrated_tp_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence']) / row['original_lc_faithfulness_divergence'],
            'Semantic Uncertainty': (row["calibrated_su_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence']) / row['original_lc_faithfulness_divergence']
        }
    else:
        ece = {
            'training_set': row['dataset'],
            'test_set': row['dataset'],
            'Linguistic Confidence': row["calibrated_lc_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE'],
            'Token Probability': row["calibrated_tp_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE'],
            'Semantic Uncertainty': row["calibrated_su_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE']
        }

        fd = {
            'training_set': row['dataset'],
            'test_set': row['dataset'],
            'Linguistic Confidence': row["calibrated_lc_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence'],
            'Token Probability': row["calibrated_tp_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence'],
            'Semantic Uncertainty': row["calibrated_su_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence']
        }
    in_domain_ece.append(ece)
    in_domain_fd.append(fd)


In [13]:
in_domain_ece_df = pd.DataFrame(in_domain_ece)
in_domain_fd_df = pd.DataFrame(in_domain_fd)

# Format latex table

In [14]:
def generate_latex_table(faithfulness_data, ece_data, pct):
    def fmt(value, pct=False):
        if value is None:
            return r"-"
        if pct:
            pct = value * 100
            color = "green!70!black" if pct < 0 else "red!70!black"
            abs_pct = abs(pct)
            return rf"\textcolor{{{color}}}{{$\Delta${abs_pct:.2f}\%}}"
        else:
            color = "green!70!black" if value < 0 else "red!70!black"
            abs_value = abs(value)
            return rf"\textcolor{{{color}}}{{$\Delta${abs_value:.4f}}}"

    def lookup(data, train, test):
        for row in data:
            if row["training_set"] == train and row["test_set"] == test:
                return row
        return None

    estimators = [
        ("Linguistic\\\\Confidence", "Linguistic Confidence"),
        ("Token\\\\Probability",     "Token Probability"),
        ("Semantic\\\\Uncertainty",  "Semantic Uncertainty"),
    ]
    datasets = ["MMLU", "SQuAD2.0", "TruthfulQA"]

    def build_block(data, metric_label):
        lines = []
        lines.append(rf"\multirow{{9}}{{=}}{{\textbf{{{metric_label}}}}}")

        for ei, (est_display, est_key) in enumerate(estimators):
            lines.append(rf"& \multirow{{3}}{{=}}{{{est_display}}}")

            for ti, train in enumerate(datasets):
                cells = []
                for col in datasets:
                    # if col == train:
                    #     cells.append("-")
                    # else:
                    row = lookup(data, train, col)
                    cells.append(fmt(row.get(est_key), pct=pct) if row else "-")

                cell_str = " & ".join(cells)

                if ti == 0:
                    lines.append(rf"& {train} & {cell_str} \\")
                else:
                    lines.append(rf"& & {train} & {cell_str} \\")

            if ei < len(estimators) - 1:
                lines.append(r"\cmidrule(lr){2-6}")

        return lines

    latex_lines = [
        r"\begin{table}[t]",
        r"\centering",
        r"\caption{Cross-domain mean linguistic-space calibration improvement results, with negative values indicating a reduction in calibration error. For each pair of datasets, we train the calibration map on one dataset and apply it to another dataset, and compute the mean change in Faithfulness Divergence and generalised ECE for each confidence estimator. The results are averaged across all models. Overall, cross-domain calibration transfers well across dataset pairs, with consistent improvements across estimators and metrics.}",
        r"\label{tab:cross-domain-calibration}",
        r"\small",
        r"\begin{tabular}{p{2cm}p{2cm}lccc}",
        r"\toprule",
        r"\textbf{Metric} & \textbf{Estimator} & \textbf{Train/Test} & {MMLU} & SQuAD2.0 & {TruthfulQA} \\",
        r"\midrule",
    ]

    latex_lines.extend(build_block(faithfulness_data, "Faithfulness\\\\Divergence\\\\Mean\\\\Reduction"))
    latex_lines.append(r"\midrule")
    latex_lines.extend(build_block(ece_data, "Generalised\\\\ECE\\\\Mean\\\\Reduction"))

    latex_lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ]

    return "\n".join(latex_lines)

print(generate_latex_table(fd_improvement_df.to_dict("records") + in_domain_fd_df.to_dict("records"), 
                           ece_improvement_df.to_dict("records") + in_domain_ece_df.to_dict("records"), 
                           pct=pct))

\begin{table}[t]
\centering
\caption{Cross-domain mean linguistic-space calibration improvement results, with negative values indicating a reduction in calibration error. For each pair of datasets, we train the calibration map on one dataset and apply it to another dataset, and compute the mean change in Faithfulness Divergence and generalised ECE for each confidence estimator. The results are averaged across all models. Overall, cross-domain calibration transfers well across dataset pairs, with consistent improvements across estimators and metrics.}
\label{tab:cross-domain-calibration}
\small
\begin{tabular}{p{2cm}p{2cm}lccc}
\toprule
\textbf{Metric} & \textbf{Estimator} & \textbf{Train/Test} & {MMLU} & SQuAD2.0 & {TruthfulQA} \\
\midrule
\multirow{9}{=}{\textbf{Faithfulness\\Divergence\\Mean\\Reduction}}
& \multirow{3}{=}{Linguistic\\Confidence}
& MMLU & \textcolor{green!70!black}{$\Delta$65.84\%} & \textcolor{green!70!black}{$\Delta$63.45\%} & \textcolor{green!70!black}{$\Delta$59.1